# ⚽ Football Predictor V2 — Leakage-Controlled Walk-Forward Betting Model

This notebook upgrades the original predictor while preserving its useful parts
(data source, Elo concept, Dixon-Coles, XGBoost, Kelly staking).

**What changed and why** is explained in the markdown cell right before each
section, and the final section gives an OLD-vs-NEW comparison so nothing is
hidden.

Sections:
1. Imports  2. Configuration  3. Data Loading  4. Data Validation
5. Team-Match History  6. Feature Engineering (unified)  7. Elo (improved)
8. Dixon-Coles (proper ξ tuning)  9. XGBoost Walk-Forward  10. Calibration (fixed)
11. Ensemble Weight Optimization  12. Score Matrix / Goal Markets
13. Asian Handicap (proper settlement)  14. No-Vig Market Pricing
15. Betting Value (EV / Edge / Fair Odds)  16. Bet Score & Decision
17. Walk-Forward Backtest  18. Closing Line Value (CLV)
19. Final Model Training  20. GUI  21. Example Prediction  22. Performance Report


## 1. Imports

In [ ]:
# Install packages
!pip install -q pandas numpy scipy scikit-learn xgboost matplotlib seaborn tqdm ipywidgets

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import poisson
from scipy.optimize import minimize
from scipy.special import softmax

from sklearn.metrics import log_loss, brier_score_loss
from sklearn.isotonic import IsotonicRegression
import xgboost as xgb

import matplotlib.pyplot as plt
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

EPS = 1e-9
print("✅ All packages ready!")


## 2. Configuration

`WINDOWS` controls the recency-weighted rolling windows (3/5/10 matches).
Everything downstream — team history, feature engineering, live GUI
predictions — reads from this single config block, so there is one source
of truth instead of magic numbers scattered through the notebook.


In [ ]:
LEAGUES = {
    "Premier League": "E0",
    "La Liga": "SP1",
    "Serie A": "I1",
    "Bundesliga": "D1",
    "Ligue 1": "F1"
}
LEAGUE_NAMES = list(LEAGUES.keys())

WINDOWS = [3, 5, 10]

STAT_COLS = ["GF", "GA", "Shots", "SOT", "Corners", "Fouls", "Yellow", "Red",
             "Points", "Win", "Draw", "Loss", "CleanSheet", "BTTS"]

DEFAULTS = {
    "GF": 1.20, "GA": 1.20, "Shots": 11.0, "SOT": 4.0, "Corners": 5.0,
    "Fouls": 11.0, "Yellow": 1.8, "Red": 0.05, "Points": 1.2,
    "Win": 0.33, "Draw": 0.27, "Loss": 0.40, "CleanSheet": 0.28, "BTTS": 0.50,
}

DC_XI_CANDIDATES = [0.0005, 0.001, 0.0015, 0.002, 0.0025, 0.003, 0.004]
ENSEMBLE_WEIGHT_GRID = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

MIN_TRAIN_SEASONS = 2   # walk-forward: need at least this many seasons before first test season


## 3. Data Loading

Preserved from the original notebook (same Football-Data.co.uk source, same
season logic, same column handling) — there was no clear technical reason to
replace working data-loading code. The only addition is filling in the
extra match-stat columns (`HF/AF/HY/AY/HR/AR`) so the new team-history table
always has something to read, and keeping the raw bookmaker odds columns
untouched for later no-vig / CLV calculations.


In [ ]:
def get_last_n_seasons(n=5):
    current_year = datetime.now().year
    if datetime.now().month < 8:
        current_year -= 1
    seasons = [f"{str(current_year - i)[-2:]}{str(current_year - i + 1)[-2:]}" for i in range(n)]
    return seasons[::-1]


def load_raw_data(n_seasons=5):
    seasons = get_last_n_seasons(n_seasons)
    print(f"📊 Loading seasons: {seasons}")

    all_data = []
    for league_name, league_code in LEAGUES.items():
        for season in seasons:
            url = f"https://www.football-data.co.uk/mmz4281/{season}/{league_code}.csv"
            try:
                d = pd.read_csv(url, encoding="latin1", on_bad_lines="skip")
                d["League"] = league_name
                d["Season"] = season
                all_data.append(d)
            except Exception:
                print(f"⚠️ Failed loading {league_name} {season}")

    df = pd.concat(all_data, ignore_index=True)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df = df.sort_values(["League", "Date"]).reset_index(drop=True)

    required = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]
    df = df.dropna(subset=required)
    df["FTHG"] = df["FTHG"].astype(int)
    df["FTAG"] = df["FTAG"].astype(int)
    df["Outcome"] = df["FTR"].map({"H": 0, "D": 1, "A": 2})

    # Match-stat columns are not present for every league/season on Football-Data;
    # fill with NaN (NOT zero) so downstream rolling means correctly skip them
    # rather than treating a missing stat as "the team had 0 shots".
    for c in ["HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF", "HY", "AY", "HR", "AR"]:
        if c not in df.columns:
            df[c] = np.nan

    print(f"✅ Loaded {len(df):,} matches")
    return df


df_raw = load_raw_data()


## 4. Data Validation

In [ ]:
def validate_dataset(df):
    print("\n==============================")
    print("DATA INTEGRITY CHECK")
    print("==============================")
    print("\nMissing values (match-stat columns):")
    print(df[["HS","AS","HST","AST","HC","AC","HF","AF","HY","AY","HR","AR"]].isna().mean().round(3))

    dup_matches = df.duplicated(subset=["Date","HomeTeam","AwayTeam","League"]).sum()
    print(f"\nDuplicate matches: {dup_matches}")

    print("\nMatches per league:")
    print(df["League"].value_counts())

    print("\nDate range:", df["Date"].min(), "→", df["Date"].max())

    future = (df["Date"] > datetime.now()).sum()
    if future > 0:
        print(f"⚠️ WARNING: {future} matches have future dates")

    print("==============================\n")


validate_dataset(df_raw)


## 5. Team-Match History Table

**Problem in V1:** rest days were computed separately for home-role and
away-role appearances (`groupby('HomeTeam')` / `groupby('AwayTeam')`), so a
team's rest was blind to matches played in the other venue role — wrong
fatigue numbers.

**Fix:** every match now produces **two rows** — one from the home team's
perspective, one from the away team's — in a single long table sorted by
`Team, Date`. Rest days, and every rolling stat, are computed on this table
so they see a team's *entire* chronological history regardless of venue.


In [ ]:
def build_team_history(df):
    home = pd.DataFrame({
        "League": df["League"], "Season": df["Season"], "Date": df["Date"],
        "Team": df["HomeTeam"], "Opponent": df["AwayTeam"], "Venue": "H",
        "GF": df["FTHG"], "GA": df["FTAG"],
        "Shots": df["HS"], "SOT": df["HST"], "Corners": df["HC"],
        "Fouls": df["HF"], "Yellow": df["HY"], "Red": df["HR"],
        "Points": np.where(df["FTR"] == "H", 3, np.where(df["FTR"] == "D", 1, 0)),
        "Win": (df["FTR"] == "H").astype(int), "Draw": (df["FTR"] == "D").astype(int),
        "Loss": (df["FTR"] == "A").astype(int),
        "CleanSheet": (df["FTAG"] == 0).astype(int),
        "BTTS": ((df["FTHG"] > 0) & (df["FTAG"] > 0)).astype(int),
    })

    away = pd.DataFrame({
        "League": df["League"], "Season": df["Season"], "Date": df["Date"],
        "Team": df["AwayTeam"], "Opponent": df["HomeTeam"], "Venue": "A",
        "GF": df["FTAG"], "GA": df["FTHG"],
        "Shots": df["AS"], "SOT": df["AST"], "Corners": df["AC"],
        "Fouls": df["AF"], "Yellow": df["AY"], "Red": df["AR"],
        "Points": np.where(df["FTR"] == "A", 3, np.where(df["FTR"] == "D", 1, 0)),
        "Win": (df["FTR"] == "A").astype(int), "Draw": (df["FTR"] == "D").astype(int),
        "Loss": (df["FTR"] == "H").astype(int),
        "CleanSheet": (df["FTHG"] == 0).astype(int),
        "BTTS": ((df["FTHG"] > 0) & (df["FTAG"] > 0)).astype(int),
    })

    hist = pd.concat([home, away], ignore_index=True)
    hist = hist.sort_values(["Team", "Date"]).reset_index(drop=True)
    return hist


team_history = build_team_history(df_raw)
print(f"✅ Team-match history built: {len(team_history):,} rows ({team_history['Team'].nunique()} teams)")


## 6. Feature Engineering — Unified Builder

**This is the most important fix in the whole notebook.**

V1's manual/GUI prediction path built features like this:

```python
team_data = df[(df['League']==league) & ((df['HomeTeam']==home)|(df['AwayTeam']==away))].tail(10)
features = team_data[feature_cols].mean().values
```

That averages a jumble of historical rows — mixing the home team's and away
team's matches together, mixing home and away roles together — and is **not**
a valid representation of "Home vs Away" as a fixture. It also has nothing to
do with the features the model was actually trained on.

**Fix:** one function, `build_match_features()`, is used for training,
walk-forward validation, OOF generation, backtesting, *and* the live GUI.
There is no separate live-prediction logic. It looks strictly backwards from
a `match_date` (no leakage), builds recency-weighted rolling stats per team —
overall and venue-specific — and derives explicit matchup features (attack
vs defence, form diff, rest diff, etc.) instead of raw averages.


In [ ]:
def _build_team_index(hist):
    idx = {}
    for team, g in hist.groupby("Team"):
        g = g.sort_values("Date")
        idx[team] = {
            "dates": g["Date"].values,
            "venue": g["Venue"].values,
            **{c: g[c].to_numpy(dtype=float) for c in STAT_COLS},
        }
    return idx


team_idx = _build_team_index(team_history)


def _weighted_mean(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan
    weights = np.linspace(1.0, 0.4, len(values))   # newest match weighted highest
    return float(np.average(values, weights=weights))


def team_stats_asof(team, cutoff_date, venue=None, windows=WINDOWS):
    """Strictly-historical (< cutoff_date) recency-weighted team stats.
    THE single function used for training features AND live predictions."""

    out = {"matches": 0}
    if team not in team_idx:
        for w in windows:
            for c in STAT_COLS:
                out[f"{c}_L{w}"] = DEFAULTS[c]
        out["rest"] = 7.0
        return out

    d = team_idx[team]
    cutoff = np.datetime64(cutoff_date)

    overall_mask = d["dates"] < cutoff
    mask = overall_mask if venue is None else (overall_mask & (d["venue"] == venue))
    positions = np.nonzero(mask)[0]
    out["matches"] = int(len(positions))

    if len(positions) == 0:
        for w in windows:
            for c in STAT_COLS:
                out[f"{c}_L{w}"] = DEFAULTS[c]
    else:
        for w in windows:
            sel = positions[-w:]
            for c in STAT_COLS:
                out[f"{c}_L{w}"] = _weighted_mean(d[c][sel])

    # Rest days: team's true chronological history, independent of `venue` arg
    overall_positions = np.nonzero(overall_mask)[0]
    if len(overall_positions) > 0:
        last_date = d["dates"][overall_positions[-1]]
        out["rest"] = float((cutoff - last_date) / np.timedelta64(1, "D"))
    else:
        out["rest"] = 7.0

    return out


def build_match_features(league, home, away, match_date):
    """Build the exact feature vector for a HOME vs AWAY fixture as of
    match_date. Used identically for training rows and live GUI predictions."""

    home_all = team_stats_asof(home, match_date, venue=None)
    home_h = team_stats_asof(home, match_date, venue="H")
    away_all = team_stats_asof(away, match_date, venue=None)
    away_a = team_stats_asof(away, match_date, venue="A")

    f = {}
    for w in WINDOWS:
        f[f"home_gf_L{w}"] = home_all[f"GF_L{w}"]
        f[f"home_ga_L{w}"] = home_all[f"GA_L{w}"]
        f[f"away_gf_L{w}"] = away_all[f"GF_L{w}"]
        f[f"away_ga_L{w}"] = away_all[f"GA_L{w}"]

        f[f"home_home_gf_L{w}"] = home_h[f"GF_L{w}"]
        f[f"home_home_ga_L{w}"] = home_h[f"GA_L{w}"]
        f[f"away_away_gf_L{w}"] = away_a[f"GF_L{w}"]
        f[f"away_away_ga_L{w}"] = away_a[f"GA_L{w}"]

        f[f"home_points_L{w}"] = home_all[f"Points_L{w}"]
        f[f"away_points_L{w}"] = away_all[f"Points_L{w}"]

        f[f"home_shots_L{w}"] = home_all[f"Shots_L{w}"]
        f[f"away_shots_L{w}"] = away_all[f"Shots_L{w}"]
        f[f"home_sot_L{w}"] = home_all[f"SOT_L{w}"]
        f[f"away_sot_L{w}"] = away_all[f"SOT_L{w}"]
        f[f"home_corners_L{w}"] = home_all[f"Corners_L{w}"]
        f[f"away_corners_L{w}"] = away_all[f"Corners_L{w}"]
        f[f"home_yellow_L{w}"] = home_all[f"Yellow_L{w}"]
        f[f"away_yellow_L{w}"] = away_all[f"Yellow_L{w}"]

    # Matchup / interaction features (5-match venue-specific window)
    f["attack_home_vs_away_defence"] = home_h["GF_L5"] - away_a["GA_L5"]
    f["attack_away_vs_home_defence"] = away_a["GF_L5"] - home_h["GA_L5"]
    f["form_diff"] = home_all["Points_L5"] - away_all["Points_L5"]
    f["shot_diff"] = home_all["Shots_L5"] - away_all["Shots_L5"]
    f["sot_diff"] = home_all["SOT_L5"] - away_all["SOT_L5"]
    f["corner_diff"] = home_all["Corners_L5"] - away_all["Corners_L5"]
    f["clean_sheet_diff"] = home_all["CleanSheet_L5"] - away_all["CleanSheet_L5"]
    f["btts_diff"] = home_all["BTTS_L5"] - away_all["BTTS_L5"]

    f["home_rest"] = home_all["rest"]
    f["away_rest"] = away_all["rest"]
    f["rest_diff"] = home_all["rest"] - away_all["rest"]

    f["home_sample"] = home_all["matches"]
    f["away_sample"] = away_all["matches"]

    for lg in LEAGUE_NAMES:
        f[f"league_{lg}"] = int(league == lg)

    return pd.Series(f)


def build_all_features(df):
    """Vectorised-as-possible application of build_match_features() to every
    historical match, in strict chronological order."""
    df = df.sort_values(["League", "Date"]).reset_index(drop=True)
    rows = [build_match_features(r["League"], r["HomeTeam"], r["AwayTeam"], r["Date"])
            for _, r in tqdm(df.iterrows(), total=len(df), desc="Building features")]
    feat_df = pd.DataFrame(rows)
    feat_df.index = df.index
    out = pd.concat([df, feat_df], axis=1)
    feature_cols = list(feat_df.columns)

    # Drop matches where either team has almost no history yet (cold start noise)
    before = len(out)
    out = out[(out["home_sample"] >= 3) & (out["away_sample"] >= 3)].reset_index(drop=True)
    print(f"✅ {len(feature_cols)} features created | {len(out):,}/{before:,} matches retained "
          f"(dropped early-season cold-start rows)")
    return out, feature_cols


df, feature_cols = build_all_features(df_raw)


## 7. Elo (Improved)

Kept the core Elo concept from V1 but:
- added a **goal-difference multiplier** (a 3-0 win moves ratings more than a 1-0 win)
- added separate **attack** and **defence** Elo tracks, not just one overall number
- still fully chronological — a team's rating at match *t* only reflects matches before *t*


In [ ]:
def build_elo(df, k=20, base=1500.0):
    data = df.sort_values("Date").copy()
    ratings, atk, dfc = {}, {}, {}
    h_elo, a_elo, h_atk, a_atk, h_def, a_def = [], [], [], [], [], []

    for _, row in data.iterrows():
        h, a = row["HomeTeam"], row["AwayTeam"]
        for t in (h, a):
            ratings.setdefault(t, base)
            atk.setdefault(t, base)
            dfc.setdefault(t, base)

        rh, ra = ratings[h], ratings[a]
        h_elo.append(rh); a_elo.append(ra)
        h_atk.append(atk[h]); a_atk.append(atk[a])
        h_def.append(dfc[h]); a_def.append(dfc[a])

        expected_h = 1 / (1 + 10 ** ((ra - rh) / 400))
        actual_h = 1.0 if row["FTR"] == "H" else (0.5 if row["FTR"] == "D" else 0.0)

        margin = abs(row["FTHG"] - row["FTAG"])
        mult = 1.0 if margin <= 1 else (1.25 if margin == 2 else (1.5 if margin == 3 else 1.75))

        change = k * mult * (actual_h - expected_h)
        ratings[h] += change
        ratings[a] -= change

        atk_change_h = k * 0.5 * mult * np.tanh((row["FTHG"] - row["FTAG"]) / 3)
        atk[h] += atk_change_h
        dfc[a] -= atk_change_h
        atk_change_a = k * 0.5 * mult * np.tanh((row["FTAG"] - row["FTHG"]) / 3)
        atk[a] += atk_change_a
        dfc[h] -= atk_change_a

    data["ELO_home"] = h_elo
    data["ELO_away"] = a_elo
    data["ELO_diff"] = data["ELO_home"] - data["ELO_away"]
    data["ELO_atk_home"] = h_atk
    data["ELO_atk_away"] = a_atk
    data["ELO_def_home"] = h_def
    data["ELO_def_away"] = a_def
    return data, ratings, atk, dfc


df, elo_ratings, elo_attack, elo_defence = build_elo(df)
feature_cols += ["ELO_diff", "ELO_atk_home", "ELO_atk_away", "ELO_def_home", "ELO_def_away"]
print("✅ Elo ratings computed (chronological, goal-margin weighted, attack/defence split)")


## 8. Dixon-Coles — Proper ξ (Time-Decay) Tuning

The model class itself (Poisson + Dixon-Coles low-score correlation
correction, time-decayed match weights) is unchanged from V1 — it was fine.

**Problem in V1:** `tune_dixon_coles_xi()` picked ξ using `abs(model.home_adv)`,
a number that has nothing to do with predictive quality.

**Fix:** genuine walk-forward selection. For each candidate ξ, fit on all
seasons before season *i*, and score the **held-out** negative log-likelihood
on season *i* (no refitting on the test season). The ξ with the lowest average
out-of-sample NLL wins.


In [ ]:
class DixonColes:
    def __init__(self, xi=0.002, max_goals=10):
        self.xi = xi
        self.max_goals = max_goals

    def fit(self, data):
        data = data.sort_values("Date").copy()
        self.teams = sorted(set(data["HomeTeam"]) | set(data["AwayTeam"]))
        n = len(self.teams)
        tidx = {t: i for i, t in enumerate(self.teams)}
        hi = data["HomeTeam"].map(tidx).values
        ai = data["AwayTeam"].map(tidx).values
        hg = data["FTHG"].values
        ag = data["FTAG"].values
        max_date = data["Date"].max()
        days = (max_date - data["Date"]).dt.days.values
        weights = np.exp(-self.xi * days)

        def nll(params):
            att = params[:n] - np.mean(params[:n])
            deff = params[n:2 * n]
            home_adv, rho = params[2 * n], params[2 * n + 1]
            lh = np.exp(home_adv + att[hi] - deff[ai])
            la = np.exp(att[ai] - deff[hi])
            p = poisson.pmf(hg, lh) * poisson.pmf(ag, la)
            corr = np.ones(len(data))
            m00 = (hg == 0) & (ag == 0); m01 = (hg == 0) & (ag == 1)
            m10 = (hg == 1) & (ag == 0); m11 = (hg == 1) & (ag == 1)
            corr[m00] = 1 - lh[m00] * la[m00] * rho
            corr[m01] = 1 + lh[m01] * rho
            corr[m10] = 1 + la[m10] * rho
            corr[m11] = 1 - rho
            p *= corr
            return -np.sum(weights * np.log(np.maximum(p, EPS)))

        x0 = np.r_[np.zeros(n), np.zeros(n), 0.2, 0.0]
        bounds = [(-3, 3)] * (2 * n) + [(-1, 1), (-0.1, 0.1)]
        res = minimize(nll, x0, method="L-BFGS-B", bounds=bounds, options={"maxiter": 300})
        p = res.x
        self.attack = dict(zip(self.teams, p[:n] - np.mean(p[:n])))
        self.defence = dict(zip(self.teams, p[n:2 * n]))
        self.home_adv, self.rho = p[2 * n], p[2 * n + 1]
        return self

    def _lambdas(self, home, away):
        lh = np.exp(self.home_adv + self.attack.get(home, 0) - self.defence.get(away, 0))
        la = np.exp(self.attack.get(away, 0) - self.defence.get(home, 0))
        return lh, la

    def score_matrix(self, home, away):
        lh, la = self._lambdas(home, away)
        g = np.arange(self.max_goals + 1)
        S = np.outer(poisson.pmf(g, lh), poisson.pmf(g, la))
        S[0, 0] *= (1 - lh * la * self.rho)
        S[0, 1] *= (1 + lh * self.rho)
        S[1, 0] *= (1 + la * self.rho)
        S[1, 1] *= (1 - self.rho)
        S = np.maximum(S, 0)
        S /= S.sum()
        return S, lh, la

    def predict(self, home, away):
        S, lh, la = self.score_matrix(home, away)
        g = np.arange(self.max_goals + 1)
        total = np.add.outer(g, g)
        return {
            "lambda_home": float(lh), "lambda_away": float(la),
            "prob_home": float(np.tril(S, -1).sum()),
            "prob_draw": float(np.trace(S)),
            "prob_away": float(np.triu(S, 1).sum()),
            "prob_over_05": float(S[total >= 1].sum()),
            "prob_over_15": float(S[total >= 2].sum()),
            "prob_over_25": float(S[total >= 3].sum()),
            "prob_over_35": float(S[total >= 4].sum()),
            "prob_under_25": float(S[total < 3].sum()),
            "btts_yes": float(S[(g[:, None] > 0) & (g[None, :] > 0)].sum()),
            "score_matrix": S,
            "exp_goals": float(lh + la),
        }

    def match_nll(self, data):
        """NLL of THIS fitted model on held-out matches (no refitting)."""
        total = 0.0
        for _, r in data.iterrows():
            lh, la = self._lambdas(r["HomeTeam"], r["AwayTeam"])
            p = poisson.pmf(r["FTHG"], lh) * poisson.pmf(r["FTAG"], la)
            hg, ag = r["FTHG"], r["FTAG"]
            corr = 1.0
            if hg == 0 and ag == 0: corr = 1 - lh * la * self.rho
            elif hg == 0 and ag == 1: corr = 1 + lh * self.rho
            elif hg == 1 and ag == 0: corr = 1 + la * self.rho
            elif hg == 1 and ag == 1: corr = 1 - self.rho
            p *= corr
            total += -np.log(max(p, EPS))
        return total


def tune_dixon_coles_xi(data, candidate_xis=DC_XI_CANDIDATES, min_train_seasons=MIN_TRAIN_SEASONS):
    """Genuine walk-forward ξ selection using held-out NLL (fixes the V1 bug
    of choosing ξ from abs(home_adv), which is unrelated to predictive fit)."""
    seasons = sorted(data["Season"].unique())
    if len(seasons) <= min_train_seasons:
        return candidate_xis[len(candidate_xis) // 2], {}

    results = {}
    for xi in candidate_xis:
        total_nll, n_eval = 0.0, 0
        for i in range(min_train_seasons, len(seasons)):
            train = data[data["Season"].isin(seasons[:i])]
            test = data[data["Season"] == seasons[i]]
            if len(train) < 20 or len(test) == 0:
                continue
            m = DixonColes(xi=xi).fit(train)
            total_nll += m.match_nll(test)
            n_eval += len(test)
        results[xi] = total_nll / max(n_eval, 1)

    best_xi = min(results, key=results.get)
    return best_xi, results


print("🔍 Tuning Dixon-Coles ξ per league (walk-forward NLL)...")
best_xi_by_league = {}
for league in LEAGUE_NAMES:
    league_data = df[df["League"] == league]
    if len(league_data) < 40:
        continue
    xi, xi_results = tune_dixon_coles_xi(league_data)
    best_xi_by_league[league] = xi
    print(f"   {league}: best ξ = {xi}  |  NLL grid = {{{', '.join(f'{k}:{v:.3f}' for k,v in xi_results.items())}}}")


## 9. XGBoost — Walk-Forward Training (Chronological OOF)

Same walk-forward-by-season structure as V1 (train on seasons `[0:i]`, test on
season `i`) — that part was already correct — but now trained on the V2
feature set (`build_match_features` output) instead of the averaged-row
features, and with `NaN` (not `0`) used for missing values so XGBoost's
native missing-value handling can do its job properly.


In [ ]:
def walk_forward_train_xgb(df, feature_cols, min_train_seasons=MIN_TRAIN_SEASONS):
    df = df.sort_values("Date").reset_index(drop=True)
    seasons = sorted(df["Season"].unique())
    oof = np.full((len(df), 3), np.nan)
    oof_valid = np.zeros(len(df), dtype=bool)
    metrics = []
    last_model = None

    for i in range(min_train_seasons, len(seasons)):
        train_idx = df["Season"].isin(seasons[:i])
        test_idx = df["Season"] == seasons[i]
        train, test = df.loc[train_idx], df.loc[test_idx]
        if len(test) == 0 or len(train) < 20:
            continue

        Xtr, ytr = train[feature_cols].values, train["Outcome"].values
        Xte, yte = test[feature_cols].values, test["Outcome"].values

        model = xgb.XGBClassifier(
            objective="multi:softprob", num_class=3, n_estimators=300,
            learning_rate=0.04, max_depth=4, min_child_weight=6,
            subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1,
            reg_lambda=2.0, gamma=0.05, random_state=42, eval_metric="mlogloss",
        )
        model.fit(Xtr, ytr)
        last_model = model
        preds = model.predict_proba(Xte)
        oof[test.index] = preds
        oof_valid[test.index] = True

        ll = log_loss(yte, preds, labels=[0, 1, 2])
        y_oh = np.zeros((len(yte), 3)); y_oh[np.arange(len(yte)), yte] = 1
        brier = np.mean(np.sum((preds - y_oh) ** 2, axis=1))
        acc = (preds.argmax(axis=1) == yte).mean()
        print(f"   📅 test={seasons[i]} | n={len(test):4d} | LogLoss={ll:.4f} | Brier={brier:.4f} | Acc={acc:.3f}")
        metrics.append({"season": seasons[i], "log_loss": ll, "brier": brier, "accuracy": acc, "n": len(test)})

    return oof, oof_valid, metrics, last_model


print("🎯 Walk-forward XGBoost training (V2 features)...")
oof_preds, oof_valid, wf_metrics_v2, _ = walk_forward_train_xgb(df, feature_cols)

valid_mask0 = oof_valid & ~np.isnan(oof_preds).any(axis=1)
wf_ll_v2 = log_loss(df.loc[valid_mask0, "Outcome"], oof_preds[valid_mask0], labels=[0,1,2])
wf_brier_v2 = np.mean(wf_metrics_v2 and [m['brier'] for m in wf_metrics_v2] or [np.nan])
print(f"\n✅ V2 walk-forward mean Log-Loss: {wf_ll_v2:.4f} | mean Brier: {np.mean([m['brier'] for m in wf_metrics_v2]):.4f}")


## 10. Calibration — Fixed OOF Validity Bug

**Problem in V1:** the OOF array was pre-allocated with `np.zeros`, and early
rows (from the first `min_train_seasons` seasons, which are never used as a
test fold) stayed at `[0,0,0]`. Calibration was then fit on **all** rows,
including those fake all-zero rows — which is almost certainly why the
V1 notebook printed a nonsensical raw log-loss of ~15.8.

**Fix:** OOF predictions are initialised to `NaN`, an explicit `oof_valid`
mask tracks which rows actually got a genuine held-out prediction, and
calibration is fit **only** on `oof_valid & not-NaN` rows.


In [ ]:
class ProbabilityCalibrator:
    def __init__(self):
        self.calibrators = []

    def fit(self, y_true, y_pred_proba):
        self.calibrators = []
        for i in range(3):
            cal = IsotonicRegression(out_of_bounds="clip")
            cal.fit(y_pred_proba[:, i], (y_true == i).astype(int))
            self.calibrators.append(cal)

    def transform(self, y_pred_proba):
        out = np.zeros_like(y_pred_proba)
        for i, cal in enumerate(self.calibrators):
            out[:, i] = cal.transform(y_pred_proba[:, i])
        out /= np.clip(out.sum(axis=1, keepdims=True), 1e-8, None)
        return out


valid_mask = oof_valid & ~np.isnan(oof_preds).any(axis=1)
y_valid = df.loc[valid_mask, "Outcome"].values
oof_valid_preds = oof_preds[valid_mask]

calibrator = ProbabilityCalibrator()
calibrator.fit(y_valid, oof_valid_preds)
calibrated_valid = calibrator.transform(oof_valid_preds)

raw_ll = log_loss(y_valid, oof_valid_preds, labels=[0, 1, 2])
cal_ll = log_loss(y_valid, calibrated_valid, labels=[0, 1, 2])

print("📊 Calibration Results (valid OOF rows ONLY)")
print(f"   Rows used: {valid_mask.sum():,} / {len(df):,}")
print(f"   Raw Log-Loss:        {raw_ll:.4f}")
print(f"   Calibrated Log-Loss: {cal_ll:.4f}")
print(f"   Improvement:         {(raw_ll - cal_ll):.4f}")

df["prob_home_raw"] = np.nan
df["prob_draw_raw"] = np.nan
df["prob_away_raw"] = np.nan
df.loc[valid_mask, "prob_home_raw"] = oof_valid_preds[:, 0]
df.loc[valid_mask, "prob_draw_raw"] = oof_valid_preds[:, 1]
df.loc[valid_mask, "prob_away_raw"] = oof_valid_preds[:, 2]


## 11. Ensemble Weight Optimization

**Problem in V1:** the Dixon-Coles/XGBoost ensemble weight was hard-coded at
`dc_weight=0.6` with no evidence it was optimal.

**Fix:** generate walk-forward OOF predictions for Dixon-Coles too (same
train-on-seasons-before-i, predict-on-season-i structure as XGBoost), then
grid-search the DC/ML blend weight on the log-pooled ensemble and keep the
value that minimises walk-forward log-loss on rows where **both** models have
a genuine OOF prediction.


In [ ]:
def dc_walk_forward_oof(df, xi_by_league, min_train_seasons=MIN_TRAIN_SEASONS):
    df = df.sort_values("Date").reset_index(drop=True)
    oof = np.full((len(df), 3), np.nan)
    valid = np.zeros(len(df), dtype=bool)

    for league, g in df.groupby("League"):
        xi = xi_by_league.get(league, 0.002)
        seasons = sorted(g["Season"].unique())
        for i in range(min_train_seasons, len(seasons)):
            train = g[g["Season"].isin(seasons[:i])]
            test = g[g["Season"] == seasons[i]]
            if len(train) < 20 or len(test) == 0:
                continue
            m = DixonColes(xi=xi).fit(train)
            for idx, row in test.iterrows():
                pred = m.predict(row["HomeTeam"], row["AwayTeam"])
                oof[idx] = [pred["prob_home"], pred["prob_draw"], pred["prob_away"]]
                valid[idx] = True
    return oof, valid


print("🎯 Generating Dixon-Coles walk-forward OOF predictions...")
dc_oof, dc_valid = dc_walk_forward_oof(df, best_xi_by_league)

both_valid = oof_valid & dc_valid & ~np.isnan(oof_preds).any(axis=1) & ~np.isnan(dc_oof).any(axis=1)
print(f"Rows with both ML and DC OOF predictions: {both_valid.sum():,}")


def optimize_ensemble_weight(y_true, ml_probs, dc_probs, weight_grid=ENSEMBLE_WEIGHT_GRID):
    results = {}
    ml_c = np.clip(ml_probs, EPS, 1 - EPS)
    dc_c = np.clip(dc_probs, EPS, 1 - EPS)
    for w in weight_grid:
        combined = softmax(w * np.log(dc_c) + (1 - w) * np.log(ml_c), axis=1)
        results[w] = log_loss(y_true, combined, labels=[0, 1, 2])
    best_w = min(results, key=results.get)
    return best_w, results


best_dc_weight, weight_results = optimize_ensemble_weight(
    df.loc[both_valid, "Outcome"].values, oof_preds[both_valid], dc_oof[both_valid]
)
print("Walk-forward log-loss by DC weight:", {k: round(v, 4) for k, v in weight_results.items()})
print(f"✅ Optimal DC weight: {best_dc_weight}  (V1 had this hard-coded at 0.6)")


def build_live_features(league, home, away, match_date, feature_cols):
    """Same build_match_features() used in training, plus the team's current
    Elo ratings (Elo is a running/stateful system, updated match-by-match
    during training — for a live fixture we simply use each team's latest
    known rating, exactly as the training loop would if this match were next)."""
    base = build_match_features(league, home, away, match_date)
    elo_extra = pd.Series({
        "ELO_diff": elo_ratings.get(home, 1500.0) - elo_ratings.get(away, 1500.0),
        "ELO_atk_home": elo_attack.get(home, 1500.0),
        "ELO_atk_away": elo_attack.get(away, 1500.0),
        "ELO_def_home": elo_defence.get(home, 1500.0),
        "ELO_def_away": elo_defence.get(away, 1500.0),
    })
    full = pd.concat([base, elo_extra])
    return full.reindex(feature_cols).values.reshape(1, -1).astype(float)


def ensemble_predict(league, home, away, match_date, dc_model, ml_model, feature_cols, dc_weight=best_dc_weight):
    dc_pred = dc_model.predict(home, away)
    dc_probs = np.clip([dc_pred["prob_home"], dc_pred["prob_draw"], dc_pred["prob_away"]], EPS, 1 - EPS)

    feats = build_live_features(league, home, away, match_date, feature_cols)
    ml_probs = np.clip(ml_model.predict_proba(feats)[0], EPS, 1 - EPS)

    combined_log = dc_weight * np.log(dc_probs) + (1 - dc_weight) * np.log(ml_probs)
    ens = softmax(combined_log)

    return {
        "prob_home": float(ens[0]), "prob_draw": float(ens[1]), "prob_away": float(ens[2]),
        "dc_prob_home": float(dc_probs[0]), "dc_prob_draw": float(dc_probs[1]), "dc_prob_away": float(dc_probs[2]),
        "ml_prob_home": float(ml_probs[0]), "ml_prob_draw": float(ml_probs[1]), "ml_prob_away": float(ml_probs[2]),
        "lambda_home": dc_pred["lambda_home"], "lambda_away": dc_pred["lambda_away"],
        "score_matrix": dc_pred["score_matrix"],
        "prob_over_05": dc_pred["prob_over_05"], "prob_over_15": dc_pred["prob_over_15"],
        "prob_over_25": dc_pred["prob_over_25"], "prob_over_35": dc_pred["prob_over_35"],
        "prob_under_25": dc_pred["prob_under_25"], "btts_yes": dc_pred["btts_yes"],
        "exp_goals": dc_pred["exp_goals"],
    }


## 12. Score Matrix & Goal Markets

The Dixon-Coles score matrix already gives us Over/Under and BTTS (Section 8).
Here we add correct-score probabilities and a compact market summary used by
the GUI and backtester.


In [ ]:
def top_correct_scores(score_matrix, n=5, max_show_goals=5):
    S = score_matrix[:max_show_goals+1, :max_show_goals+1]
    flat = [((i, j), S[i, j]) for i in range(S.shape[0]) for j in range(S.shape[1])]
    flat.sort(key=lambda x: -x[1])
    return flat[:n]


def market_probabilities(pred):
    probs = {
        "home": pred["prob_home"], "draw": pred["prob_draw"], "away": pred["prob_away"],
        "over_0.5": pred["prob_over_05"], "over_1.5": pred["prob_over_15"],
        "over_2.5": pred["prob_over_25"], "over_3.5": pred["prob_over_35"],
        "under_2.5": pred["prob_under_25"], "btts_yes": pred["btts_yes"],
        "btts_no": 1 - pred["btts_yes"],
        "1X": pred["prob_home"] + pred["prob_draw"],
        "X2": pred["prob_draw"] + pred["prob_away"],
        "12": pred["prob_home"] + pred["prob_away"],
        "dnb_home": pred["prob_home"] / max(1 - pred["prob_draw"], EPS),
        "dnb_away": pred["prob_away"] / max(1 - pred["prob_draw"], EPS),
    }
    return probs


print("✅ Score-matrix goal markets ready (O/U 0.5-3.5, BTTS, DNB, double chance, correct score)")


## 13. Asian Handicap — Proper Score-Matrix Settlement

**Problem in V1:**

```python
goal_diff = pred["lambda_home"] - pred["lambda_away"]
prob_home_cover = 1 / (1 + np.exp(-goal_diff))
```

A logistic squash of the expected-goal difference is **not** the probability
of covering a handicap line — it ignores the actual goal-difference
distribution and can't represent pushes at all.

**Fix:** settle every line directly against the Dixon-Coles score matrix,
handling **half** wins/losses for quarter lines (e.g. -0.25, +0.75) by
averaging the two neighbouring standard lines, and pushes (stake returned)
for whole lines.


In [ ]:
def asian_handicap_settlement(score_matrix, line, side, max_goals=10):
    """side: 'home' or 'away'. `line` is the handicap applied to `side`.
    Returns win/push/loss probability mass, settled directly from the score
    matrix (quarter lines = average of the two neighbouring standard lines)."""

    g = np.arange(max_goals + 1)
    diff = np.subtract.outer(g, g)  # home_goals - away_goals, shape (home, away)

    def settle(single_line):
        adj = (diff + single_line) if side == "home" else (-diff + single_line)
        win = score_matrix[adj > 0].sum()
        push = score_matrix[adj == 0].sum()
        loss = score_matrix[adj < 0].sum()
        return win, push, loss

    is_quarter = np.isclose(abs(line * 4) % 1, 0.5)
    if is_quarter:
        l1, l2 = line - 0.25, line + 0.25
        w1, p1, lo1 = settle(l1)
        w2, p2, lo2 = settle(l2)
        win, push, loss = (w1 + w2) / 2, (p1 + p2) / 2, (lo1 + lo2) / 2
    else:
        win, push, loss = settle(line)

    return {"win": win, "push": push, "loss": loss}


def asian_handicap_ev(score_matrix, line, side, odds, max_goals=10):
    s = asian_handicap_settlement(score_matrix, line, side, max_goals)
    # Push returns stake (0 P/L); win pays (odds-1) per unit; loss = -1 per unit
    ev = s["win"] * (odds - 1) + s["push"] * 0 - s["loss"] * 1
    win_prob_effective = s["win"] / max(s["win"] + s["loss"], EPS)  # for reporting
    return {"win": s["win"], "push": s["push"], "loss": s["loss"], "EV": ev,
            "win_prob_effective": win_prob_effective}


AH_LINES = [-1.25, -1.0, -0.75, -0.5, -0.25, 0.25, 0.5, 0.75, 1.0, 1.25]
print("✅ Asian Handicap settlement engine ready (proper win/push/loss from score matrix)")


## 14. No-Vig Market Pricing

Bookmaker odds always contain overround (the bookmaker's margin). Comparing
model probability to *raw* implied probability (`1/odds`) systematically
understates the model's edge. We strip the overround out first.


In [ ]:
def no_vig_probs(odds_list):
    odds = np.array(odds_list, dtype=float)
    raw = 1 / odds
    overround = raw.sum() - 1
    fair = raw / raw.sum()
    return fair, overround


print("✅ No-vig pricing ready")


## 15. Betting Value — EV, Edge, Fair Odds

In [ ]:
def value_metrics(model_prob, odds, market_fair_prob=None):
    if odds is None or pd.isna(odds) or odds <= 1:
        return None
    implied = 1 / odds
    ev = model_prob * odds - 1
    out = {
        "model_probability": model_prob, "odds": odds, "implied_probability": implied,
        "fair_odds": (1 / model_prob) if model_prob > 0 else np.inf,
        "EV": ev, "EV_pct": ev * 100,
    }
    if market_fair_prob is not None:
        out["market_fair_probability"] = market_fair_prob
        out["model_edge_vs_market"] = model_prob - market_fair_prob
    return out


print("✅ EV / edge / fair-odds calculator ready")


## 16. Bet Score & Decision Logic

A transparent 0-100 score combining probability, EV, market edge, model
agreement (DC vs XGBoost) and data quality (sample size), then a 3-way
decision: `BET` / `LEAN` / `NO BET`. Thresholds are deliberately conservative
region-based (not a single overfit combination) — see Section 17 for how they
were checked against walk-forward data.


In [ ]:
BET_SCORE_WEIGHTS = {"probability": 25, "ev": 30, "edge": 20, "agreement": 15, "data_quality": 10}

def model_agreement(dc_prob, ml_prob):
    """1.0 = models fully agree, 0.0 = maximal disagreement (e.g. DC 70% vs ML 43%)."""
    return float(max(0.0, 1.0 - abs(dc_prob - ml_prob) / 0.5))


def data_quality_score(home_sample, away_sample, min_matches=10):
    return float(min(1.0, min(home_sample, away_sample) / min_matches))


def calculate_bet_score(probability, ev, edge, agreement, data_quality):
    score = 0
    score += min(probability / 0.80, 1) * BET_SCORE_WEIGHTS["probability"]
    score += min(max(ev, 0) / 0.15, 1) * BET_SCORE_WEIGHTS["ev"]
    score += min(max(edge, 0) / 0.10, 1) * BET_SCORE_WEIGHTS["edge"]
    score += agreement * BET_SCORE_WEIGHTS["agreement"]
    score += data_quality * BET_SCORE_WEIGHTS["data_quality"]
    return round(min(score, 100), 1)


def classify_bet(probability, ev, edge, score):
    if probability >= 0.65 and ev >= 0.05 and edge >= 0.03 and score >= 70:
        return "🟢 BET"
    if probability >= 0.55 and ev >= 0.02 and edge >= 0.015 and score >= 55:
        return "🟡 LEAN"
    return "⚪ NO BET"


print("✅ Bet Score + BET/LEAN/NO BET decision logic ready")


## 17. Walk-Forward Backtest (True Out-of-Sample)

The backtest only ever uses **out-of-fold** probabilities — the same
`oof_preds` (XGBoost) and `dc_oof` (Dixon-Coles) walk-forward arrays from
Sections 9 & 11 — combined at the optimal weight found in Section 11, then
calibrated using the Section 10 calibrator. No probability used for staking a
historical bet was produced by a model that had seen that match's outcome.

Kelly staking is fractional (quarter-Kelly by default) with per-bet and daily
exposure caps, matching V1's risk-management concept.


In [ ]:
class RiskManager:
    def __init__(self, bankroll=10000, kelly_frac=0.25, max_bet_pct=0.05,
                 max_daily_pct=0.20, max_bets_per_match=3, min_edge=0.015,
                 drawdown_stop=0.25):
        self.initial_bankroll = bankroll
        self.bankroll = bankroll
        self.peak_bankroll = bankroll
        self.kelly_frac = kelly_frac
        self.max_bet_pct = max_bet_pct
        self.max_daily_pct = max_daily_pct
        self.max_bets_per_match = max_bets_per_match
        self.min_edge = min_edge
        self.drawdown_stop = drawdown_stop
        self.daily_exposure = 0
        self.match_bets = {}
        self.stopped = False

    def kelly_stake(self, prob, odds, edge, match_id=None):
        if self.stopped or edge < self.min_edge or prob <= 0.05 or prob >= 0.95 or odds <= 1.0:
            return 0
        b = odds - 1
        kelly = (prob * b - (1 - prob)) / b
        if kelly <= 0:
            return 0
        kelly = min(kelly * self.kelly_frac, self.max_bet_pct)
        stake = self.bankroll * kelly
        if self.daily_exposure + stake > self.bankroll * self.max_daily_pct:
            return 0
        if match_id is not None:
            if self.match_bets.get(match_id, 0) >= self.max_bets_per_match:
                return 0
            self.match_bets[match_id] = self.match_bets.get(match_id, 0) + 1
        stake = round(stake, 2)
        self.daily_exposure += stake
        return stake

    def record_bet(self, stake, won, odds):
        profit = stake * (odds - 1) if won else -stake
        self.bankroll += profit
        self.peak_bankroll = max(self.peak_bankroll, self.bankroll)
        drawdown = (self.peak_bankroll - self.bankroll) / self.peak_bankroll
        if drawdown >= self.drawdown_stop:
            self.stopped = True
        return profit

    def reset_daily(self):
        self.daily_exposure = 0
        self.match_bets = {}


def run_walk_forward_backtest(df, oof_ml, oof_dc, valid_ml, valid_dc, calibrator,
                                dc_weight, min_prob=0.55, min_ev=0.03, max_odds=6.0,
                                bankroll=10000):
    both_valid_local = np.asarray(valid_ml) & np.asarray(valid_dc) & ~np.isnan(oof_ml).any(axis=1) & ~np.isnan(oof_dc).any(axis=1)
    sub = df.loc[both_valid_local].copy().reset_index(drop=True)
    ml = np.clip(oof_ml[both_valid_local], EPS, 1 - EPS)
    dcp = np.clip(oof_dc[both_valid_local], EPS, 1 - EPS)

    # keep sub and the probability arrays aligned, then sort all three together chronologically
    order = np.argsort(sub["Date"].values)
    sub = sub.iloc[order].reset_index(drop=True)
    ml, dcp = ml[order], dcp[order]

    combined = softmax(dc_weight * np.log(dcp) + (1 - dc_weight) * np.log(ml), axis=1)
    combined = calibrator.transform(combined)
    risk = RiskManager(bankroll=bankroll)
    trades = []
    prev_date = None

    for i in range(len(sub)):
        row = sub.iloc[i]
        if prev_date is not None and row["Date"] != prev_date:
            risk.reset_daily()
        prev_date = row["Date"]

        match_id = f"{row['Date']}_{row['HomeTeam']}_{row['AwayTeam']}"
        probs = combined[i]

        markets = {
            "home": (probs[0], row.get("AvgH"), "H"),
            "draw": (probs[1], row.get("AvgD"), "D"),
            "away": (probs[2], row.get("AvgA"), "A"),
        }

        for market, (prob, odds, outcome) in markets.items():
            if odds is None or pd.isna(odds) or odds <= 1 or odds > max_odds:
                continue
            if prob < min_prob:
                continue
            implied = 1 / odds
            ev = prob * odds - 1
            edge = prob - implied
            if ev < min_ev:
                continue

            stake = risk.kelly_stake(prob, odds, edge, match_id)
            if stake <= 0:
                continue

            won = row["FTR"] == outcome
            profit = risk.record_bet(stake, won, odds)

            closing_col = {"home": "MaxH", "draw": "MaxD", "away": "MaxA"}[market]
            closing_odds = row.get(closing_col)
            clv = (closing_odds - odds) / odds if closing_odds and not pd.isna(closing_odds) else None

            trades.append({
                "date": row["Date"], "league": row["League"],
                "match": f"{row['HomeTeam']} vs {row['AwayTeam']}", "market": market,
                "prob": prob, "odds": odds, "ev": ev, "edge": edge, "stake": stake,
                "won": won, "profit": profit, "bankroll": risk.bankroll, "clv": clv,
            })

    trades_df = pd.DataFrame(trades)
    if len(trades_df) == 0:
        return trades_df, {"n_trades": 0, "note": "No trades passed filters"}

    winners = trades_df[trades_df["won"]]
    losers = trades_df[~trades_df["won"]]
    stats = {
        "initial": bankroll, "final": risk.bankroll,
        "profit": risk.bankroll - bankroll,
        "roi_pct": (risk.bankroll - bankroll) / bankroll * 100,
        "yield_pct": trades_df["profit"].sum() / trades_df["stake"].sum() * 100,
        "n_trades": len(trades_df), "win_rate": len(winners) / len(trades_df) * 100,
        "avg_odds": trades_df["odds"].mean(), "avg_prob": trades_df["prob"].mean() * 100,
        "avg_ev": trades_df["ev"].mean() * 100, "avg_edge": trades_df["edge"].mean() * 100,
        "profit_factor": (winners["profit"].sum() / abs(losers["profit"].sum())) if len(losers) > 0 else None,
        "max_drawdown_pct": ((risk.peak_bankroll - trades_df["bankroll"].min()) / risk.peak_bankroll) * 100,
        "avg_clv": trades_df["clv"].mean(skipna=True),
        "positive_clv_pct": (trades_df["clv"] > 0).mean() * 100 if trades_df["clv"].notna().any() else None,
    }
    return trades_df, stats


print("⚡ Running NEW-model walk-forward backtest (out-of-fold probabilities only)...")
trades_df_v2, stats_v2 = run_walk_forward_backtest(
    df, oof_preds, dc_oof, oof_valid, dc_valid, calibrator, best_dc_weight,
    min_prob=0.55, min_ev=0.03,
)
print("\n==============================")
print("NEW MODEL — WALK-FORWARD BACKTEST")
print("==============================")
for k, v in stats_v2.items():
    print(f"{k:18}: {v:.3f}" if isinstance(v, float) else f"{k:18}: {v}")


### 17b. Threshold Sensitivity (Robust Region, Not a Single Overfit Point)

V1 hard-coded `min_prob=0.55, min_ev=0.08` and got zero trades — then the
obvious next step would have been to keep lowering thresholds until trades
appeared, which is exactly how you overfit a backtest. Instead we scan a
small grid on the same out-of-fold data and look for a **robust region**
(nearby settings giving similar results), not a single cherry-picked point.


In [ ]:
prob_grid = [0.50, 0.55, 0.60, 0.65]
ev_grid = [0.02, 0.03, 0.04, 0.05, 0.06, 0.08]

grid_results = []
for mp in prob_grid:
    for me in ev_grid:
        _, s = run_walk_forward_backtest(df, oof_preds, dc_oof, oof_valid, dc_valid,
                                          calibrator, best_dc_weight, min_prob=mp, min_ev=me)
        grid_results.append({"min_prob": mp, "min_ev": me, **{k: s.get(k) for k in
                             ["n_trades", "roi_pct", "yield_pct", "max_drawdown_pct"]}})

grid_df = pd.DataFrame(grid_results)
print(grid_df.to_string(index=False))


## 18. Closing Line Value (CLV)

In [ ]:
if len(trades_df_v2) > 0 and trades_df_v2["clv"].notna().any():
    print("📊 CLV Diagnostics")
    print(f"   Average CLV:      {trades_df_v2['clv'].mean():.4f}")
    print(f"   Median CLV:       {trades_df_v2['clv'].median():.4f}")
    print(f"   Positive CLV %:   {(trades_df_v2['clv'] > 0).mean()*100:.1f}%")
    print("\n   CLV by league:")
    print(trades_df_v2.groupby("league")["clv"].mean().round(4))
    print("\n   CLV by market:")
    print(trades_df_v2.groupby("market")["clv"].mean().round(4))
else:
    print("No trades with CLV data available (check that MaxH/MaxD/MaxA columns are populated).")


## 19. Final Model Training (for live/GUI use)

For live predictions we need one final XGBoost model and one Dixon-Coles
model per league, both trained on **all** available history (there's no
"future" left to hold out once we're predicting a genuinely upcoming match).

Note this final in-sample log-loss is **not** a performance claim — it's just
a fit check. The number to trust for "how good is this model really" is the
walk-forward log-loss from Section 9/17, which is what gets reported in
Section 22.


In [ ]:
print("🚀 Training final models on full history...")

X_full = df[feature_cols].values
y_full = df["Outcome"].values

final_ml_model = xgb.XGBClassifier(
    objective="multi:softprob", num_class=3, n_estimators=300,
    learning_rate=0.04, max_depth=4, min_child_weight=6,
    subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1,
    reg_lambda=2.0, gamma=0.05, random_state=42, eval_metric="mlogloss",
)
final_ml_model.fit(X_full, y_full)
print(f"✅ Final XGBoost trained on {len(df):,} matches "
      f"(in-sample log-loss={log_loss(y_full, final_ml_model.predict_proba(X_full)):.4f} "
      f"— NOT the number to trust, see Section 22)")

final_dc_models = {}
for league in df["League"].unique():
    xi = best_xi_by_league.get(league, 0.002)
    m = DixonColes(xi=xi).fit(df[df["League"] == league])
    final_dc_models[league] = m
    print(f"   ✅ {league}: {len(m.teams)} teams, ξ={xi}, home_adv={m.home_adv:.3f}, ρ={m.rho:.3f}")

print(f"\n✅ {len(final_dc_models)} Dixon-Coles models ready | final calibrator fit on {valid_mask.sum():,} valid OOF rows")


## 20. GUI

Same manual-entry concept as V1 (paste `League, Home, Away, HomeOdds,
DrawOdds, AwayOdds` lines) — but the prediction path now calls
`ensemble_predict()`, which internally calls the **exact same**
`build_match_features()` used for training. There is no separate averaging
logic. Asian Handicap odds are optional extra columns; if provided they're
settled via the score matrix (Section 13), not the old logistic hack.


In [ ]:
manual_matches = []
predictions_results = []


def create_styled_box(title, children, color="#2c3e50"):
    header = widgets.HTML(value=f'''
        <div style="background-color: {color}; padding: 15px; border-radius: 8px 8px 0 0; margin-top: 10px;">
            <h3 style="color: white; margin: 0;">{title}</h3>
        </div>''')
    content = widgets.VBox(children=children, layout=widgets.Layout(
        border=f'2px solid {color}', border_radius='0 0 8px 8px', padding='15px', background='#f8f9fa'))
    return widgets.VBox([header, content])


paste_area = widgets.Textarea(
    value='',
    placeholder='League, Home, Away, HomeOdds, DrawOdds, AwayOdds\n'
                'Premier League, Arsenal, Chelsea, 2.05, 3.50, 4.10',
    layout=widgets.Layout(width='100%', height='150px', font_family='monospace'))

parse_button = widgets.Button(description='📋 Parse Games', button_style='primary',
                               layout=widgets.Layout(width='150px', height='40px'))
clear_button = widgets.Button(description='🗑️ Clear', button_style='warning',
                               layout=widgets.Layout(width='100px', height='40px'))
predict_button = widgets.Button(description='🔮 GENERATE PREDICTIONS', button_style='success',
                                 layout=widgets.Layout(width='300px', height='50px'))

status_output = widgets.Output()
results_output = widgets.Output()


def parse_games(b):
    global manual_matches
    manual_matches = []
    with status_output:
        clear_output()
        text = paste_area.value.strip()
        if not text:
            print("❌ No games pasted!")
            return
        parsed, errors = 0, []
        for line in text.split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            try:
                parts = [p.strip() for p in line.split(',')]
                if len(parts) != 6:
                    errors.append(f"Wrong format: {line[:50]}")
                    continue
                manual_matches.append({
                    'league': parts[0], 'home': parts[1], 'away': parts[2],
                    'odds_home': float(parts[3]), 'odds_draw': float(parts[4]), 'odds_away': float(parts[5]),
                })
                parsed += 1
            except Exception:
                errors.append(f"Error parsing: {line[:30]}...")
        print(f"✅ Parsed {parsed} matches")
        for err in errors[:3]:
            print(f"   ⚠️ {err}")


def clear_paste(b):
    global manual_matches
    paste_area.value = ''
    manual_matches = []
    with status_output:
        clear_output(); print("🗑️ Cleared")
    with results_output:
        clear_output()


def predict_all(b):
    global predictions_results
    predictions_results = []
    with status_output:
        clear_output()
    with results_output:
        clear_output()

    if not manual_matches:
        with status_output:
            print("❌ No matches! Parse games first.")
        return

    today = pd.Timestamp(datetime.now())

    with results_output:
        for match in manual_matches:
            league, home, away = match['league'], match['home'], match['away']
            odds = {"home": match['odds_home'], "draw": match['odds_draw'], "away": match['odds_away']}

            if league not in final_dc_models:
                print(f"⚠️ {home} vs {away}: unknown league '{league}', skipping"); continue
            dc_model = final_dc_models[league]
            if home not in dc_model.teams or away not in dc_model.teams:
                print(f"⚠️ {home} vs {away}: team not recognised in {league}, skipping"); continue

            pred = ensemble_predict(league, home, away, today, dc_model, final_ml_model, feature_cols)
            fair, overround = no_vig_probs([odds['home'], odds['draw'], odds['away']])

            agreement = np.mean([
                model_agreement(pred['dc_prob_home'], pred['ml_prob_home']),
                model_agreement(pred['dc_prob_draw'], pred['ml_prob_draw']),
                model_agreement(pred['dc_prob_away'], pred['ml_prob_away']),
            ])
            home_stats = team_stats_asof(home, today)
            away_stats = team_stats_asof(away, today)
            dq = data_quality_score(home_stats['matches'], away_stats['matches'])

            print("=" * 66)
            print(f"{home} vs {away}  ({league})")
            print("=" * 66)
            print(f"{'':16}{'Home':>10}{'Draw':>10}{'Away':>10}")
            print(f"{'Dixon-Coles':16}{pred['dc_prob_home']*100:9.1f}%{pred['dc_prob_draw']*100:9.1f}%{pred['dc_prob_away']*100:9.1f}%")
            print(f"{'XGBoost':16}{pred['ml_prob_home']*100:9.1f}%{pred['ml_prob_draw']*100:9.1f}%{pred['ml_prob_away']*100:9.1f}%")
            print(f"{'Ensemble':16}{pred['prob_home']*100:9.1f}%{pred['prob_draw']*100:9.1f}%{pred['prob_away']*100:9.1f}%")
            print(f"\nModel agreement: {agreement*100:.0f}/100   Data quality: {dq*100:.0f}/100")
            print(f"Expected goals — Home: {pred['lambda_home']:.2f}  Away: {pred['lambda_away']:.2f}")
            print(f"O/U 2.5: over {pred['prob_over_25']*100:.1f}% | under {pred['prob_under_25']*100:.1f}%   "
                  f"BTTS yes: {pred['btts_yes']*100:.1f}%")

            print(f"\nMarket (no-vig, overround {overround*100:.1f}%):  "
                  f"Home {fair[0]*100:.1f}%  Draw {fair[1]*100:.1f}%  Away {fair[2]*100:.1f}%")

            best_bet, best_score = None, -1
            for sel, prob_key, fair_key in [("Home", 'prob_home', 0), ("Draw", 'prob_draw', 1), ("Away", 'prob_away', 2)]:
                mprob = pred[prob_key]
                o = odds[sel.lower()]
                vm = value_metrics(mprob, o, market_fair_prob=fair[fair_key])
                if vm is None:
                    continue
                sc = calculate_bet_score(mprob, vm['EV'], vm['model_edge_vs_market'], agreement, dq)
                if sc > best_score:
                    best_score = sc
                    best_bet = (sel, mprob, o, vm, sc)

            if best_bet:
                sel, mprob, o, vm, sc = best_bet
                decision = classify_bet(mprob, vm['EV'], vm['model_edge_vs_market'], sc)
                print(f"\nBest value: {sel}  |  Model {mprob*100:.1f}% vs market {vm['market_fair_probability']*100:.1f}% "
                      f"(edge {vm['model_edge_vs_market']*100:+.1f}%)")
                print(f"Fair odds {vm['fair_odds']:.2f}  |  Available {o:.2f}  |  EV {vm['EV_pct']:+.1f}%  |  "
                      f"Bet Score {sc}/100")
                print(f"DECISION: {decision}")
            print()

            predictions_results.append({'match': f"{home} vs {away}", 'league': league, 'pred': pred})

    with status_output:
        print(f"✅ Processed {len(predictions_results)} matches")


parse_button.on_click(parse_games)
clear_button.on_click(clear_paste)
predict_button.on_click(predict_all)

display(HTML('''
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 22px; border-radius: 12px; margin-bottom: 16px;">
    <h2 style="color: white; margin: 0;">⚽ FOOTBALL PREDICTOR V2</h2>
    <p style="color: #f0f0f0; margin: 4px 0 0 0; font-size: 13px;">Leakage-controlled · walk-forward validated · unified feature builder</p>
</div>'''))

display(create_styled_box("📋 STEP 1: PASTE GAMES", [
    widgets.HTML('<div style="background:#fff3cd;padding:10px;border-radius:5px;margin-bottom:10px;border-left:4px solid #ffc107;">'
                 '<strong>Format:</strong> League, Home, Away, HomeOdds, DrawOdds, AwayOdds</div>'),
    paste_area, widgets.HBox([parse_button, clear_button]), status_output
], color="#667eea"))

display(create_styled_box("🔮 STEP 2: PREDICT", [predict_button], color="#764ba2"))
display(create_styled_box("📊 RESULTS", [results_output], color="#28a745"))

print("✅ GUI loaded")


## 21. Example Prediction

In [ ]:
# Quick sanity-check prediction using two teams and a league present in df
_example_league = df['League'].iloc[-1]
_example_row = df[df['League'] == _example_league].iloc[-1]
_home, _away = _example_row['HomeTeam'], _example_row['AwayTeam']

_pred = ensemble_predict(_example_league, _home, _away, pd.Timestamp(datetime.now()),
                          final_dc_models[_example_league], final_ml_model, feature_cols)

print(f"Example fixture: {_home} vs {_away} ({_example_league})")
print(f"  Ensemble  — Home {_pred['prob_home']*100:.1f}%  Draw {_pred['prob_draw']*100:.1f}%  Away {_pred['prob_away']*100:.1f}%")
print(f"  Expected goals — Home {_pred['lambda_home']:.2f}  Away {_pred['lambda_away']:.2f}")
print(f"  Top correct scores: {top_correct_scores(_pred['score_matrix'], n=3)}")

_ah = asian_handicap_settlement(_pred['score_matrix'], -0.5, 'home')
print(f"  AH -0.5 Home settlement — win {_ah['win']*100:.1f}%  push {_ah['push']*100:.1f}%  loss {_ah['loss']*100:.1f}%")


## 22. Performance Report — OLD vs NEW

To make the comparison honest rather than asserted, this section re-runs the
**original V1 feature-engineering function** (`create_features_v1`, copied
verbatim from your uploaded notebook's Cell 5 — it already used `shift(1)`
correctly, so it is not itself leaky) through the exact same walk-forward
XGBoost harness used for V2 (Section 9). That isolates the effect of the
feature-engineering/Elo/rest-day changes from the walk-forward methodology,
which was already sound in both.

The other V1 bugs (calibration using invalid OOF rows, ξ chosen from
`abs(home_adv)`, fixed 0.6 ensemble weight, logistic Asian Handicap, and
live-prediction row-averaging) are **not** re-run here because reproducing a
known bug just to print a number it would have produced isn't informative —
they're already fixed in Sections 10, 8, 11, 13 and 6 respectively, each with
before/after reasoning at the point of the fix.


In [ ]:
def create_features_v1(df):
    """Verbatim (behaviourally) reproduction of the ORIGINAL notebook's
    create_features(), for a fair walk-forward comparison against V2."""
    df = df.copy().sort_values(['League', 'Date']).reset_index(drop=True)

    for col in ['HS', 'AS', 'HST', 'AST', 'HC', 'AC']:
        if col not in df.columns:
            df[col] = 0
    df[['HS', 'AS', 'HST', 'AST', 'HC', 'AC']] = df[['HS', 'AS', 'HST', 'AST', 'HC', 'AC']].fillna(0)

    for window in [5, 10]:
        df[f'HGS_L{window}'] = df.groupby(['League', 'HomeTeam'])['FTHG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        df[f'HGC_L{window}'] = df.groupby(['League', 'HomeTeam'])['FTAG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        df[f'AGS_L{window}'] = df.groupby(['League', 'AwayTeam'])['FTAG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        df[f'AGC_L{window}'] = df.groupby(['League', 'AwayTeam'])['FTHG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean())

    df['HS_L5'] = df.groupby(['League', 'HomeTeam'])['HS'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df['AS_L5'] = df.groupby(['League', 'AwayTeam'])['AS'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df['HST_L5'] = df.groupby(['League', 'HomeTeam'])['HST'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df['AST_L5'] = df.groupby(['League', 'AwayTeam'])['AST'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df['HC_L5'] = df.groupby(['League', 'HomeTeam'])['HC'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    df['AC_L5'] = df.groupby(['League', 'AwayTeam'])['AC'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

    df['HP'] = (df['FTR'] == 'H') * 3 + (df['FTR'] == 'D') * 1
    df['AP'] = (df['FTR'] == 'A') * 3 + (df['FTR'] == 'D') * 1
    df['HForm'] = df.groupby(['League', 'HomeTeam'])['HP'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
    df['AForm'] = df.groupby(['League', 'AwayTeam'])['AP'].transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())

    # The ORIGINAL (buggy) rest-day calc: separate home-role / away-role history
    df['HomeRest'] = df.groupby(['League', 'HomeTeam'])['Date'].diff().dt.days
    df['AwayRest'] = df.groupby(['League', 'AwayTeam'])['Date'].diff().dt.days
    df['RestDiff'] = df['HomeRest'] - df['AwayRest']

    df['AttackDiff'] = df['HGS_L5'] - df['AGC_L5']
    df['DefenseDiff'] = df['HGC_L5'] - df['AGS_L5']
    df['ShotDiff'] = df['HS_L5'] - df['AS_L5']
    df['ShotTargetDiff'] = df['HST_L5'] - df['AST_L5']
    df['CornerDiff'] = df['HC_L5'] - df['AC_L5']

    if 'ELO_home' in df.columns:
        df['ELO_diff'] = df['ELO_home'] - df['ELO_away']
    else:
        df['ELO_diff'] = 0.0

    league_dummies = pd.get_dummies(df['League'], prefix='Lg')
    df = pd.concat([df, league_dummies], axis=1)

    feature_cols_v1 = [c for c in df.columns if ('_L' in c or 'Form' in c or 'Rest' in c
                                                    or 'Diff' in c or 'ELO' in c or 'Lg_' in c)]
    df = df.dropna(subset=feature_cols_v1)
    return df, feature_cols_v1


print("🔁 Reproducing V1 pipeline (original features) for a fair baseline comparison...")
_df_v1_base, _, _, _ = build_elo(df_raw.copy())  # same Elo call V1's data loader made
df_v1, feature_cols_v1 = create_features_v1(_df_v1_base)
print(f"V1-style features: {len(feature_cols_v1)} | matches retained: {len(df_v1):,}")

print("\n🎯 Walk-forward XGBoost on V1 features...")
oof_v1, valid_v1, wf_metrics_v1, _ = walk_forward_train_xgb(df_v1, feature_cols_v1)
mask_v1 = valid_v1 & ~np.isnan(oof_v1).any(axis=1)
wf_ll_v1 = log_loss(df_v1.loc[mask_v1, 'Outcome'], oof_v1[mask_v1], labels=[0, 1, 2])
wf_brier_v1 = np.mean([m['brier'] for m in wf_metrics_v1]) if wf_metrics_v1 else np.nan
wf_acc_v1 = np.mean([m['accuracy'] for m in wf_metrics_v1]) if wf_metrics_v1 else np.nan

wf_ll_v2_final = log_loss(df.loc[valid_mask, 'Outcome'], oof_preds[valid_mask], labels=[0, 1, 2])
wf_brier_v2_final = np.mean([m['brier'] for m in wf_metrics_v2])
wf_acc_v2_final = np.mean([m['accuracy'] for m in wf_metrics_v2])


In [ ]:
print("=" * 70)
print("MODEL VALIDATION REPORT")
print("=" * 70)

report_rows = [
    ("Walk-forward Log-Loss (XGBoost only)", wf_ll_v1, wf_ll_v2_final),
    ("Walk-forward Brier",                    wf_brier_v1, wf_brier_v2_final),
    ("Walk-forward Accuracy",                  wf_acc_v1, wf_acc_v2_final),
]

print(f"{'Metric':38}{'OLD':>12}{'NEW':>12}{'CHANGE':>12}")
for name, old, new in report_rows:
    change = new - old
    arrow = "▼ better" if change < 0 and "Log-Loss" in name or "Brier" in name else ("▲ better" if change > 0 else "")
    print(f"{name:38}{old:12.4f}{new:12.4f}{change:+12.4f}")

print("\nCalibration (Section 10):")
print(f"  {'Raw OOF Log-Loss (valid rows only)':38}{raw_ll:12.4f}")
print(f"  {'Calibrated Log-Loss':38}{cal_ll:12.4f}")

print("\nDixon-Coles ξ selection (Section 8): now chosen by walk-forward NLL, not abs(home_adv)")
for lg, xi in best_xi_by_league.items():
    print(f"  {lg:20}: ξ = {xi}")

print(f"\nEnsemble DC weight (Section 11): OLD fixed at 0.6  →  NEW optimized to {best_dc_weight}")

print("\nWalk-forward backtest (Section 17, out-of-fold probabilities only):")
for k, v in stats_v2.items():
    print(f"  {k:18}: {v:.3f}" if isinstance(v, float) else f"  {k:18}: {v}")

print("\n" + "=" * 70)
if wf_ll_v2_final < wf_ll_v1:
    print(f"✅ V2 improved walk-forward log-loss by {wf_ll_v1 - wf_ll_v2_final:.4f} vs V1's feature set.")
else:
    print(f"⚠️ V2 did NOT improve walk-forward log-loss on this run "
          f"({wf_ll_v2_final:.4f} vs {wf_ll_v1:.4f}). On a small/synthetic sample this can "
          f"happen by chance — re-check on the full multi-season real dataset before trusting it.")
print("=" * 70)


### Remaining Limitations

- **Shot/corner/card stats are missing for some leagues/seasons** on
  Football-Data — those rows fall back to the `DEFAULTS` in Section 2 rather
  than being fabricated, but that does dilute their signal in early seasons.
- **No real xG.** The shot/SOT-based proxy is a reasonable stand-in but is
  not a substitute for provider xG data if you can obtain it later.
- **Dixon-Coles is fit per league**, so it can't currently learn cross-league
  effects (e.g. two teams from different leagues in a cup match).
- **Backtest odds are historical closing/average prices**, not the exact
  price you'd have gotten in real time — CLV gives a sense of how far off
  that is, but it's not a live execution simulation.
- **This model is not shown to be profitable** unless the walk-forward ROI /
  CLV numbers above are positive on the *full* real dataset — treat the
  synthetic-data numbers used to build/verify this notebook as a functional
  smoke test only, not evidence of edge.
